# 05: Automated Monthly Churn Brief Pipeline

**Goal:** Build a production pipeline that simulates monthly data arrival,
computes subscription KPIs, detects statistical anomalies, identifies at-risk user
cohorts, and auto-generates two types of stakeholder briefs via the Anthropic API.

**Why two brief types?**
Different stakeholders need different outputs from the same underlying data:
- **Monthly Ops Brief** -- for a VP or director: what happened this month, what's
  anomalous, and what needs attention (1 page, plain language)
- **At-Risk Cohort Alert** -- for a retention team: which specific user behaviors
  are spiking right now, grounded in the churn driver analysis from NB04

**What makes this more than a basic API call:**
- Structured prompt engineering (system + user separation, format instructions)
- NB04 churn driver findings embedded as domain context in every prompt
- Tone control: exec-summary mode vs. full analyst mode via a single flag
- Modular pipeline design: each stage is a testable function

**Input:** `transactions_v2.csv`, `train_v2.csv`  
**Output:** Monthly briefs saved as markdown files to `briefs/`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = "../../data/"
BRIEF_DIR = "../../briefs/"
os.makedirs(BRIEF_DIR, exist_ok=True)

In [ ]:
# Anthropic API setup

api_key = os.environ.get("ANTHROPIC_API_KEY", "")
HAS_API_KEY = bool(api_key)

if HAS_API_KEY:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic API key found -- live brief generation enabled.")
else:
    print("No ANTHROPIC_API_KEY set -- pipeline will run fully but save prompts instead of briefs.")
    print("Set key with: export ANTHROPIC_API_KEY='sk-ant-...'") 

## 1. Load and Prepare Data

In [ ]:
txn = pd.read_csv(
    DATA_DIR + "transactions_v2.csv",
    dtype={
        "msno": str, "payment_method_id": int, "payment_plan_days": int,
        "plan_list_price": int, "actual_amount_paid": int, "is_auto_renew": int,
        "transaction_date": str, "membership_expire_date": str, "is_cancel": int,
    },
)
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"], format="%Y%m%d")
txn["membership_expire_date"] = pd.to_datetime(txn["membership_expire_date"], format="%Y%m%d", errors="coerce")
txn["txn_month"] = txn["transaction_date"].dt.to_period("M")

print(f"Transactions: {len(txn):,}")
print(f"Date range: {txn['transaction_date'].min().date()} to {txn['transaction_date'].max().date()}")

In [ ]:
train = pd.read_csv(DATA_DIR + "train_v2.csv", dtype={"msno": str, "is_churn": int})
churn_set = set(train[train["is_churn"] == 1]["msno"])
print(f"Churn labels: {len(train):,} users  |  Churned: {len(churn_set):,}")

## 2. Churn Driver Context (from NB04)

I embed the key findings from the logistic regression analysis directly into our
prompts. This grounds the LLM's recommendations in the actual data rather than
generic subscription advice.

In [ ]:
# These are the NB04 results - update after running NB04 on your data.
# Format: (feature label, odds_ratio, plain-English interpretation)

CHURN_DRIVERS = [
    ("Days since last transaction",     15.29, "Users inactive before expiry are 15x more likely to churn"),
    ("Number of cancellations",          4.92, "Each prior cancellation nearly 5x the churn odds"),
    ("Avg discount received",            1.48, "Discount-reliant users show higher churn risk"),
    ("Active listening days (last 30d)", 1.21, "Low recent engagement predicts churn"),
    ("Auto-renew rate",                  0.45, "Auto-renew users are 55% less likely to churn"),
    ("2+ cancellations flag",            0.73, "The flag dampens individual cancel signal once captured"),
]

CHURN_DRIVER_CONTEXT = "\n".join([
    f"  - {label}: OR={or_:.2f} -- {note}"
    for label, or_, note in CHURN_DRIVERS
])

print("Churn driver context loaded:")
print(CHURN_DRIVER_CONTEXT)

## 3. Monthly KPI Engine

In [ ]:
KPI_COLS = [
    "n_transactions", "n_unique_users", "auto_renew_rate", "cancel_rate",
    "n_cancellations", "avg_revenue", "total_revenue", "discount_rate",
    "pct_30d_plans", "non_renew_rate", "churn_pressure",
]

def compute_monthly_kpis(txn_df, month_period):
    """Compute KPIs for a single month of transaction data."""
    m = txn_df[txn_df["txn_month"] == month_period].copy()
    if len(m) == 0:
        return None

    auto_renew_rate = m["is_auto_renew"].mean()
    cancel_rate = m["is_cancel"].mean()
    non_renew_rate = 1 - auto_renew_rate

    return {
        "month": str(month_period),
        "n_transactions": len(m),
        "n_unique_users": m["msno"].nunique(),
        "auto_renew_rate": round(auto_renew_rate, 4),
        "cancel_rate": round(cancel_rate, 4),
        "n_cancellations": int(m["is_cancel"].sum()),
        "avg_revenue": round(m["actual_amount_paid"].mean(), 2),
        "total_revenue": int(m["actual_amount_paid"].sum()),
        "discount_rate": round((m["actual_amount_paid"] < m["plan_list_price"]).mean(), 4),
        "pct_30d_plans": round((m["payment_plan_days"] == 30).mean(), 4),
        "non_renew_rate": round(non_renew_rate, 4),
        "churn_pressure": round((cancel_rate + non_renew_rate) / 2, 4),
    }

In [ ]:
# Compute KPIs for all months (exclude Feb-Mar 2017 dataset cutoff artifacts)

months = sorted(txn["txn_month"].unique())
months = [m for m in months if str(m) not in ("2017-02", "2017-03")]
kpi_records = [compute_monthly_kpis(txn, m) for m in months]
kpi_df = pd.DataFrame([r for r in kpi_records if r])

print(f"Monthly KPIs computed for {len(kpi_df)} months")
kpi_df.tail(8)

## 4. Anomaly Detection

Compare each month's KPIs against a rolling 3-month baseline.
Flag any metric where |z-score| > 2.

In [ ]:
def detect_anomalies(kpi_df, current_idx, lookback=3, z_threshold=2.0):
    """Return dict of anomalous KPIs with z-score and % change vs. rolling baseline."""
    if current_idx < lookback:
        return {}

    current = kpi_df.iloc[current_idx]
    baseline = kpi_df.iloc[current_idx - lookback : current_idx]
    anomalies = {}

    for col in KPI_COLS:
        val = current[col]
        if val is None or pd.isna(val):
            continue
        base_vals = baseline[col].dropna()
        if len(base_vals) < 2:
            continue
        mean, std = base_vals.mean(), base_vals.std()
        if std < 1e-9:
            continue
        z = (val - mean) / std
        pct_change = ((val - mean) / mean * 100) if mean != 0 else 0
        if abs(z) > z_threshold:
            anomalies[col] = {
                "value": val,
                "baseline_mean": round(mean, 4),
                "z_score": round(z, 2),
                "pct_change": round(pct_change, 1),
                "direction": "increased" if z > 0 else "decreased",
            }

    return anomalies

In [ ]:
# Preview anomaly scan

print("Anomaly scan (final 7 months):\n")
for i in range(max(3, len(kpi_df) - 7), len(kpi_df)):
    month = kpi_df.iloc[i]["month"]
    anom = detect_anomalies(kpi_df, i)
    status = f"{len(anom)} anomaly/anomalies" if anom else "nominal"
    print(f"  {month}: {status}")
    for k, v in anom.items():
        print(f"    {k}: {v['direction']} {abs(v['pct_change']):.1f}% (z={v['z_score']:+.2f})")

## 5. At-Risk Cohort Detection

Beyond aggregate KPIs, I identified which *specific user behaviors* are spiking
this month, grounded in the churn drivers from NB04. This is what goes into
the At-Risk Cohort Alert brief.

In [ ]:
def detect_at_risk_cohorts(txn_df, month_period):
    """
    Flag user cohorts showing elevated churn signals this month.
    Thresholds are derived from the NB04 churn driver analysis.
    """
    m = txn_df[txn_df["txn_month"] == month_period].copy()
    if len(m) == 0:
        return {}, 0

    total_users = m["msno"].nunique()
    cohorts = {}

    # Risk Signal 1: Users with multiple cancellations this month (NB04: OR=4.9)
    multi_cancel = m.groupby("msno")["is_cancel"].sum()
    repeat_cancellers = (multi_cancel >= 2).sum()
    if repeat_cancellers > 0:
        cohorts["repeat_cancellers"] = {
            "n_users": int(repeat_cancellers),
            "pct_of_month": round(repeat_cancellers / total_users * 100, 1),
            "signal": "2+ cancellation transactions",
            "driver_or": 4.92,
            "action": "Immediate outreach -- strong churn predictor (OR=4.92)",
        }

    # Risk Signal 2: Users who switched off auto-renew (NB04: auto-renew OR=0.45)
    latest_ar = m.sort_values("transaction_date").groupby("msno")["is_auto_renew"].last()
    no_autorenew = (latest_ar == 0).sum()
    pct_no_ar = no_autorenew / total_users
    if pct_no_ar > 0.15:  # flag if >15% of this month's users have auto-renew off
        cohorts["no_auto_renew"] = {
            "n_users": int(no_autorenew),
            "pct_of_month": round(pct_no_ar * 100, 1),
            "signal": "Auto-renew disabled on most recent transaction",
            "driver_or": 0.45,
            "action": "Auto-renew re-engagement campaign -- 55% risk reduction if enrolled",
        }

    # Risk Signal 3: High-discount users (NB04: avg_discount OR=1.48)
    discounted = m[m["actual_amount_paid"] < m["plan_list_price"]]
    discount_users = discounted["msno"].nunique()
    if discount_users > 0:
        cohorts["discount_dependent"] = {
            "n_users": int(discount_users),
            "pct_of_month": round(discount_users / total_users * 100, 1),
            "signal": "Transacted at discounted price",
            "driver_or": 1.48,
            "action": "Monitor -- discount-reliant users show 48% higher churn odds",
        }

    return cohorts, total_users

## 6. Prompt Engineering

Two brief types, two tone modes. The system prompt establishes the analyst
persona once; the user prompt carries the data and instructions.

In [ ]:
SYSTEM_PROMPT = (
    "You are a senior data analyst at KKBox, Asia's largest music streaming service. "
    "You write clear, data-driven briefs for internal stakeholders. "
    "Your writing is direct, uses specific numbers, and leads with the most important insight. "
    "You never pad with filler phrases like 'it\'s worth noting' or 'it is important to mention.' "
    "Bullet points are fine. Avoid jargon."
)

def build_ops_brief_prompt(month_kpis, anomalies, prev_kpis=None, mode="full"):
    """
    Monthly Ops Brief prompt.
    mode: 'exec' = 3-sentence summary | 'full' = complete brief with sections
    """
    kpi_lines = "\n".join([f"  {k}: {v}" for k, v in month_kpis.items() if k != "month"])

    if anomalies:
        anomaly_lines = "\n".join([
            f"  - {k}: {v['direction']} {abs(v['pct_change']):.1f}% vs. 3-month avg "
            f"(current={v['value']}, z={v['z_score']:+.2f})"
            for k, v in anomalies.items()
        ])
    else:
        anomaly_lines = "  None -- all KPIs within normal range."

    mom_lines = ""
    if prev_kpis:
        changes = []
        for k in KPI_COLS:
            curr, prev = month_kpis.get(k), prev_kpis.get(k)
            if curr and prev and prev != 0:
                changes.append(f"  {k}: {(curr-prev)/prev*100:+.1f}% MoM")
        if changes:
            mom_lines = "MONTH-OVER-MONTH CHANGES:\n" + "\n".join(changes)

    if mode == "exec":
        format_instruction = (
            "Write a 3-sentence executive summary only. Lead with the single most important "
            "finding. End with one recommendation. No headers, no bullets."
        )
    else:
        format_instruction = (
            "Write a structured brief with these sections:\n"
            "1. Executive Summary (2-3 sentences)\n"
            "2. Key Metrics (bullet points with context, not just numbers)\n"
            "3. Anomalies & Risks (explain what each anomaly means for the business)\n"
            "4. Recommendations (2-3 specific, actionable items with expected impact)"
        )

    return (
        f"REPORTING MONTH: {month_kpis['month']}\n\n"
        f"KEY PERFORMANCE INDICATORS:\n{kpi_lines}\n\n"
        f"STATISTICAL ANOMALIES (vs. 3-month rolling baseline):\n{anomaly_lines}\n\n"
        f"{mom_lines}\n\n"
        f"TASK: {format_instruction}"
    )


def build_cohort_alert_prompt(month, cohorts, total_users, anomalies):
    """
    At-Risk Cohort Alert prompt.
    Uses NB04 churn driver findings as grounding context.
    """
    if not cohorts:
        return None

    cohort_lines = []
    for name, info in cohorts.items():
        cohort_lines.append(
            f"  Cohort: {name}\n"
            f"    Users: {info['n_users']:,} ({info['pct_of_month']}% of active users)\n"
            f"    Signal: {info['signal']}\n"
            f"    Churn model context: OR={info['driver_or']:.2f}\n"
            f"    Suggested action: {info['action']}"
        )

    anomaly_summary = ""
    if anomalies:
        top = list(anomalies.items())[:3]
        anomaly_summary = "Concurrent KPI anomalies this month: " + ", ".join(
            [f"{k} {v['direction']} {abs(v['pct_change']):.0f}%" for k, v in top]
        )

    return (
        f"RETENTION TEAM ALERT -- {month}\n\n"
        f"KNOWN CHURN DRIVERS (from logistic regression analysis, OR = odds ratio):\n"
        f"{CHURN_DRIVER_CONTEXT}\n\n"
        f"AT-RISK COHORTS DETECTED THIS MONTH (total active users: {total_users:,}):\n"
        + "".join(cohort_lines) + "\n\n"
        f"{anomaly_summary}\n\n"
        "TASK: Write a focused retention alert for the team. Include:\n"
        "1. Which cohort is the highest priority and why (reference the OR values)\n"
        "2. A specific outreach strategy for each cohort (what channel, what message angle)\n"
        "3. What success looks like -- what metric to track over the next 30 days\n"
        "Keep it under 300 words. Be specific and action-oriented."
    )

## 7. Pipeline: Simulate Monthly Data Arrival

Process the last 6 months of the dataset as if each month's data just arrived.
For each month: compute KPIs -> detect anomalies -> identify at-risk cohorts
-> generate both brief types -> save as markdown.

In [ ]:
def call_api(system_prompt, user_prompt):
    """Call Anthropic API. Returns (brief_text, input_tokens, output_tokens)."""
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )
    usage = response.usage
    return response.content[0].text, usage.input_tokens, usage.output_tokens


def save_brief(filepath, month, brief_type, content, is_prompt=False):
    """Save a brief or prompt to a markdown file."""
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"# {brief_type} -- {month}\n")
        f.write(f"*Generated {datetime.now().strftime('%Y-%m-%d %H:%M')}*\n\n")
        if is_prompt:
            f.write("**[No API key -- prompt shown below]**\n\n---\n\n")
            f.write(f"**SYSTEM:**\n```\n{SYSTEM_PROMPT}\n```\n\n")
            f.write(f"**USER:**\n```\n{content}\n```\n")
        else:
            f.write(content)

In [ ]:
# Simulation window: last 6 complete months
SIM_START = max(3, len(kpi_df) - 6)
sim_indices = list(range(SIM_START, len(kpi_df)))
sim_months = [kpi_df.iloc[i]["month"] for i in sim_indices]

print(f"Running pipeline for: {sim_months}\n")

results = {}
total_input_tokens = 0
total_output_tokens = 0

for i in sim_indices:
    month = kpi_df.iloc[i]["month"]
    month_kpis = kpi_df.iloc[i].to_dict()
    prev_kpis = kpi_df.iloc[i - 1].to_dict() if i > 0 else None
    anomalies = detect_anomalies(kpi_df, i)
    cohorts, n_users = detect_at_risk_cohorts(txn, kpi_df.iloc[i]["month"])

    # --- Brief Type 1: Monthly Ops Brief ---
    ops_prompt = build_ops_brief_prompt(month_kpis, anomalies, prev_kpis, mode="full")

    if HAS_API_KEY:
        ops_brief, in_tok, out_tok = call_api(SYSTEM_PROMPT, ops_prompt)
        total_input_tokens += in_tok
        total_output_tokens += out_tok
    else:
        ops_brief = None

    save_brief(
        os.path.join(BRIEF_DIR, f"{month}_ops_brief.md"),
        month, "Monthly Ops Brief",
        ops_brief if ops_brief else ops_prompt,
        is_prompt=(ops_brief is None),
    )

    # --- Brief Type 2: At-Risk Cohort Alert ---
    cohort_prompt = build_cohort_alert_prompt(month, cohorts, n_users, anomalies)
    cohort_brief = None

    if cohort_prompt:
        if HAS_API_KEY:
            cohort_brief, in_tok, out_tok = call_api(SYSTEM_PROMPT, cohort_prompt)
            total_input_tokens += in_tok
            total_output_tokens += out_tok
        save_brief(
            os.path.join(BRIEF_DIR, f"{month}_cohort_alert.md"),
            month, "At-Risk Cohort Alert",
            cohort_brief if cohort_brief else cohort_prompt,
            is_prompt=(cohort_brief is None),
        )

    results[month] = {
        "kpis": month_kpis,
        "anomalies": anomalies,
        "cohorts": cohorts,
        "ops_brief": ops_brief,
        "cohort_brief": cohort_brief,
        "ops_prompt": ops_prompt,
        "cohort_prompt": cohort_prompt,
    }

    print(f"  {month}: {len(anomalies)} anomalies, {len(cohorts)} at-risk cohorts -- briefs saved")

if HAS_API_KEY:
    print(f"\nAPI usage: {total_input_tokens:,} input tokens, {total_output_tokens:,} output tokens")
print(f"Briefs saved to: {BRIEF_DIR}")

## 8. Display Latest Month's Output

In [ ]:
latest_month = sim_months[-1]
latest = results[latest_month]

print(f"{'=' * 65}")
print(f"  MONTHLY OPS BRIEF -- {latest_month}")
print(f"{'=' * 65}\n")

if latest["ops_brief"]:
    print(latest["ops_brief"])
else:
    print("[No API key -- prompt below]\n")
    print(latest["ops_prompt"])

In [ ]:
if latest["cohort_prompt"]:
    print(f"\n{'=' * 65}")
    print(f"  AT-RISK COHORT ALERT -- {latest_month}")
    print(f"{'=' * 65}\n")

    if latest["cohort_brief"]:
        print(latest["cohort_brief"])
    else:
        print("[No API key -- prompt below]\n")
        print(latest["cohort_prompt"])

## 9. Exec Summary Mode Demo

The same KPIs, a tighter prompt instruction, a very different output.
Demonstrating prompt engineering: tone and format are controllable parameters.

In [ ]:
exec_prompt = build_ops_brief_prompt(
    latest["kpis"], latest["anomalies"], mode="exec"
)

print("EXEC SUMMARY PROMPT (mode='exec'):\n")
print(exec_prompt)

if HAS_API_KEY:
    exec_brief, _, _ = call_api(SYSTEM_PROMPT, exec_prompt)
    print(f"\nEXEC SUMMARY OUTPUT:\n{'=' * 50}")
    print(exec_brief)
else:
    print("\n[Set ANTHROPIC_API_KEY to see the exec summary output vs. the full brief]")

## 10. KPI Trend Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    "Monthly KPI Trends -- KKBox Subscriptions\n(Jan 2015 -- Jan 2017)",
    fontsize=15, fontweight="bold", y=1.01
)

plot_config = [
    ("n_transactions",  "Transaction Volume",       "#2196F3"),
    ("auto_renew_rate", "Auto-Renew Rate",           "#4CAF50"),
    ("cancel_rate",     "Cancellation Rate",         "#F44336"),
    ("avg_revenue",     "Avg Revenue / Transaction", "#FF9800"),
    ("churn_pressure",  "Churn Pressure Index",      "#9C27B0"),
    ("n_unique_users",  "Unique Active Users",       "#00BCD4"),
]

for ax, (col, title, color) in zip(axes.flat, plot_config):
    vals = kpi_df[col].values
    x = range(len(vals))
    ax.plot(x, vals, marker="o", markersize=3.5, color=color, linewidth=2)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xticks(list(x)[::4])
    ax.set_xticklabels([kpi_df.iloc[j]["month"] for j in list(x)[::4]], rotation=45, fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR + "monthly_kpi_trends.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Pipeline Summary

In [ ]:
print(f"{'=' * 65}")
print(f"  SIMULATION SUMMARY")
print(f"{'=' * 65}\n")
print(f"{'Month':<12} {'Anomalies':>10} {'At-Risk Cohorts':>16}  Top Anomaly")
print("-" * 65)
for month, data in results.items():
    n_anom = len(data["anomalies"])
    n_coh = len(data["cohorts"])
    top = list(data["anomalies"].items())[0] if data["anomalies"] else None
    top_str = f"{top[0]}: {top[1]['direction']} {abs(top[1]['pct_change']):.0f}%" if top else "no anomalies"
    print(f"{month:<12} {n_anom:>10} {n_coh:>16}  {top_str}")

In [ ]:
print(f"\n{'=' * 65}")
print("NOTEBOOK 05 -- PIPELINE SUMMARY")
print(f"{'=' * 65}")
print(f"Months processed:     {len(results)}")
print( "Brief types:          2 (Ops Brief + Cohort Alert)")
print( "Tone modes:           2 (exec summary + full analyst)")
print(f"API mode:             {'LIVE (claude-sonnet-4-20250514)' if HAS_API_KEY else 'OFFLINE (prompts saved)'}")
print(f"Total briefs saved:   {sum(2 if r['cohort_prompt'] else 1 for r in results.values())}")
print(f"Output dir:           {BRIEF_DIR}")
print( "\nPipeline stages:")
print( "  [1] Transaction data  ->  Monthly KPI engine")
print( "  [2] Rolling baseline  ->  Z-score anomaly detection")
print( "  [3] Transaction sigs  ->  At-risk cohort identification")
print( "  [4] NB04 OR context   ->  Prompt grounding")
print( "  [5] Anthropic API     ->  Brief generation (2 types x 2 tones)")
print( "  [6] Markdown files    ->  Stakeholder delivery")
print(f"{'=' * 65}")